# Atropos quickstart - first verdict in five minutes

Atropos kills quantitative hypotheses cheaply: a signal enters a 7-stage
adversarial pipeline (null controls -> specification curve -> robustness
battery -> walk-forward -> Monte Carlo risk -> predictive eval ->
cross-sectional tests) and leaves with exactly one verdict.

**Setup** (run once, outside the notebook):

```bash
uv sync        # or: pip install -e .
```

Python 3.11+; dependencies are numpy, pandas, scipy only. Make sure this
notebook runs on the project environment's kernel.

In this notebook we preregister one hypothesis, build two synthetic
worlds - one with a genuine edge, one where the "new" signal is just
disguised momentum - and get a verdict for each in a single call.


In [ ]:
import atropos
import numpy as np
import pandas as pd
from atropos import generate_prereg, run_full_pipeline

print("atropos", atropos.__version__)

## 1. Two synthetic worlds

`run_full_pipeline` takes a plain dict: `returns` (per-period forward
returns) and `signal` on a common index, optionally `features` (known
control factors the signal must beat *incrementally*). Panel mode
(`panel_signal` / `panel_returns` DataFrames, rows = dates, columns =
assets) enables the cross-sectional stage - not used here.

World A plants a hidden persistent driver inside the signal. World B is
the classic repackaging trap: a "fresh" signal that is really observed
momentum plus noise, while the pipeline already sees momentum in
`features`.


In [ ]:
rng = np.random.default_rng(4)
n_obs = 900

# latent AR(1) states: persistent momentum and a hidden driver,
# independent of each other by construction (separate shocks)
mom = np.zeros(n_obs)
driver = np.zeros(n_obs)
mom_shocks = rng.standard_normal(n_obs)
driver_shocks = rng.standard_normal(n_obs)
for pos in range(1, n_obs):
    mom[pos] = 0.5 * mom[pos - 1] + mom_shocks[pos]
    driver[pos] = 0.5 * driver[pos - 1] + driver_shocks[pos]
mom /= mom.std()
driver /= driver.std()

# world A: the signal carries the hidden driver (plus momentum bleed)
# -> a genuine incremental edge
world_edge = {
    "signal": driver + 0.3 * mom,
    "returns": driver + 0.6 * mom + 0.55 * rng.standard_normal(n_obs),
}

# world B: the "fresh" signal is really observed momentum + noise,
# and the pipeline already observes momentum in `features`
world_echo = {
    "signal": mom + 0.4 * rng.standard_normal(n_obs),
    "returns": 0.6 * mom + 0.55 * rng.standard_normal(n_obs),
    "features": pd.DataFrame(
        {"mom_noisy": mom + 0.10 * rng.standard_normal(n_obs)}
    ),
}
print("worlds built:", sorted(world_edge), "| rows:", n_obs)

## 2. Preregister before the data

Doctrine: the preregistration is frozen *before* any results are looked
at. `generate_prereg` validates a params dict (name, hypothesis,
construction, nulls, kill criteria, power calc, spec hash) and renders
frozen markdown. Kill criteria must be hard directional rules on
numbers - that is enforced, not encouraged.


In [ ]:
prereg_doc = generate_prereg(
    {
        "name": "driver_edge",
        "hypothesis": (
            "a persistent order-flow state predicts the next-period "
            "return, incrementally to momentum"
        ),
        "construction": "long/flat on zscore(signal), one-period hold",
        "nulls": [
            "no incremental predictive power vs momentum",
            "the effect is an artifact (placebo / concentration)",
        ],
        "kill_criteria": [
            {
                "criterion": "|t| of the signal after momentum controls",
                "comparison": "<",
                "threshold": 2.0,
            }
        ],
        "power_calc": {"N": n_obs},
        "hash": "quickstart-01",  # in real use: hash of the spec/code
    },
    date="2026-11-03",  # explicit date keeps the notebook reproducible
)
print("\n".join(prereg_doc.splitlines()[:16]))

## 3. One call, one verdict per hypothesis

Stages attack in adversarial order and the first FAIL kills; everything
after it is SKIPPED. Verdict names: `SURVIVES`, `SURVIVES_WITH_WARNINGS`,
`REPACKAGING`, `ARTIFACT`, `DEAD`, `NO_OOS_EDGE`, `COST_DEATH`, `RISK`,
`CS_NO_EDGE`.


In [ ]:
results = {}
for label, world in [
    ("genuine edge", world_edge),
    ("momentum echo", world_echo),
]:
    results[label] = run_full_pipeline(world)

for label, res in results.items():
    verdict = res["overall_verdict"]
    killer = res["killed_by_stage"]
    suffix = f"  (killed at stage {killer})" if killer else ""
    print(f"{label:<14}-> {verdict}{suffix}")

## 4. Reading the result

The survivor shows every stage it passed (stage 7 is SKIPPED without
panel data - it is the optional cross-sectional stage). The echo never
got past stage 1: the null controls recognized that it adds nothing
beyond known momentum. That is the framework working as intended -
killing a bad hypothesis in milliseconds, before any capital is spent.

`SURVIVES` is not the end of the story: in production, compare the live
PnL against the prereg with `atropos.track_record.live_verdict`
(`LIVE_OK` / `LIVE_KILLED` / `INCONCLUSIVE`).


In [ ]:
res = results["genuine edge"]
print("verdict:", res["overall_verdict"], "| n_obs:", res["n_obs"])
for stage in res["stages"]:
    print(f"  {stage['stage']}  {stage['name']:<24}{stage['status']}")

echo = results["momentum echo"]
print("\necho verdict:", echo["overall_verdict"])
print("echo COD:", echo["kill_reason"])

## Read next

- `docs/how-to/` - step-by-step guides
- `GLOSSARY.md` - prereg, kill criteria, COD, graveyard, DSR/PSR
- `FRAMEWORK.md` sections 1-3 - philosophy, the 7-stage pipeline, the six laws
- `KNOWN-ISSUES.md` - every skip and xfail, and why they are honest
